In [ ]:
# Sprint 53 Task 2 — Multi-Prompt LLM-as-a-Judge Experiment
# This notebook adapts the Sprint 51 Qwen3-8B judge/calibration workflow into a controlled prompt-engineering experiment.
# It evaluates: 
# - P0 baseline 
# - P1 compliance checklist 
# - P2 semantic-priority 
# - P3 error-first / MQM-style 
# - P4 combined structured prompt
# - 50-example human-labeled EN→Mandarin gold set
# - targeted Sprint 53 Task 1 challenge set
# - Phase A validity comparison at temperature 0
# - Phase B repeated-run consistency stress test at temperature 0.2
# - CSV/JSON outputs and final prompt-comparison tables

In [1]:
# Cell 1 — Install dependencies

import importlib.util, subprocess, sys
if importlib.util.find_spec("unsloth") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 9.0 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports

import os, re, gc, json, math, random, statistics, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score
from transformers import set_seed
from unsloth import FastLanguageModel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1635: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Cell 3 — Configuration

CONFIG = {
    "experiment_name": "sprint53_task2_multiprompt_judge",
    "seed": 53,
    "data": {
        
        "gold_set_csv_path": "/kaggle/input/datasets/abdighaz/gold-set/gold_set_labeled_chatgpt.csv",
        "direction": "EN_CMN",
        "max_gold_samples": 50,
    },
    "judge": {
        "model_name": "Qwen/Qwen3-8B",
        "max_seq_length": 4096,
        "max_new_tokens": 512,
        "load_in_4bit": True,
        "phase_a_temperature": 0.0,
        "phase_a_runs": 1,
        "phase_b_temperature": 0.2,
        "phase_b_runs": 5,
        "base_generation_seed": 5300,
    },
    "selection": {"top_prompts_for_phase_b": 3},
    "consistency": {
        "high_disagreement_range": 2.0,
        "high_disagreement_std": 1.0,
    },
    "output": {"root": "/kaggle/working/sprint53_task2_outputs"},
    "dry_run": False,
    "dry_run_n": 5,
}

PROMPT_IDS = [
    "P0_baseline",
    "P1_compliance",
    "P2_semantic",
    "P3_error_first",
    "P4_combined",
]

OUT = Path(CONFIG["output"]["root"])
for folder in [
    OUT, OUT/"raw_runs", OUT/"aggregated", OUT/"summaries",
    OUT/"challenge", OUT/"tables"
]:
    folder.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print("Output:", OUT)

Output: /kaggle/working/sprint53_task2_outputs


In [4]:
# Cell 4 — Helpers

def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def safe_float(x):
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None

In [5]:
# Cell 5 — Gold-set loader

@dataclass
class GoldSample:
    sample_id: str
    direction: str
    source_en: str
    mt_output: str
    overall_gold: float
    reference_zh: Optional[str] = None

def _first_existing_column(df, names, required=True):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for name in names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    if required:
        raise KeyError(f"Missing one of {names}. Available: {list(df.columns)}")
    return None

def load_gold_set(path: str) -> List[GoldSample]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Gold CSV not found: {path}\n"
            "Update CONFIG['data']['gold_set_csv_path']."
        )
    df = pd.read_csv(path)

    id_col = _first_existing_column(df, ["sample_id","id","example_id","segment_id"])
    dir_col = _first_existing_column(df, ["direction","translation_direction","lang_direction"], False)
    src_col = _first_existing_column(df, ["source_en","english","source","source_text","src"])
    mt_col = _first_existing_column(df, ["mt_output","mandarin","translation","candidate","hypothesis","output"])
    gold_col = _first_existing_column(df, ["overall_gold","gold_overall","human_score","gold_score","overall"])
    ref_col = _first_existing_column(df, ["reference_zh","reference","reference_translation","target_reference"], False)

    samples = []
    for _, row in df.iterrows():
        gold = safe_float(row[gold_col])
        if gold is None:
            continue
        samples.append(GoldSample(
            sample_id=str(row[id_col]),
            direction=str(row[dir_col]).strip() if dir_col else CONFIG["data"]["direction"],
            source_en=str(row[src_col]),
            mt_output=str(row[mt_col]),
            overall_gold=gold,
            reference_zh=None if ref_col is None or pd.isna(row[ref_col]) else str(row[ref_col]),
        ))
    return samples

def validate_gold_set(samples):
    issues, seen, counts = [], set(), {}
    for s in samples:
        counts[s.direction] = counts.get(s.direction, 0) + 1
        if s.sample_id in seen:
            issues.append(f"Duplicate sample_id: {s.sample_id}")
        seen.add(s.sample_id)
        if not 1 <= s.overall_gold <= 10:
            issues.append(f"{s.sample_id}: gold score out of range")
        if not s.source_en.strip():
            issues.append(f"{s.sample_id}: empty source")
        if not s.mt_output.strip():
            issues.append(f"{s.sample_id}: empty candidate")
    return {"n_total":len(samples),"counts":counts,"issues":issues}

In [6]:
# Cell 6 — Sprint 51 aggregation + agreement utilities

SCORE_MIN, SCORE_MAX = 1.0, 10.0
ZERO_OUTLIER_STD_THRESHOLD = 2.0
HIGH_DISAGREEMENT_STD = 1.5
MIN_VALID_RUNS_FOR_SCORE = 2
MALFORMED_RUN_FAILURE_FRACTION = 0.5
GEMBA_LAMBDA = 0.5

@dataclass
class RunResult:
    score: Optional[float]
    parse_ok: bool
    raw_response: Optional[str] = None

def _is_out_of_range(score):
    return not (SCORE_MIN <= score <= SCORE_MAX)

def aggregate_runs(runs):
    valid_scores, malformed_count = [], 0
    n_total = len(runs)

    for r in runs:
        if not r.parse_ok or r.score is None or _is_out_of_range(r.score):
            malformed_count += 1
        else:
            valid_scores.append(float(r.score))

    malformed_fraction = malformed_count / n_total if n_total else 1.0

    # Extension for Phase A single-run prompt comparison.
    if n_total == 1:
        if len(valid_scores) == 1:
            s = valid_scores[0]
            return {
                "final_score":round(s,3),"confidence":1.0,"flag":"ok",
                "aggregation_method":"single_run","n_total_runs":1,"n_valid_runs":1,
                "n_used_in_final":1,"malformed_fraction":0.0,
                "flagged_outlier_scores":[],"median":round(s,3),"std":0.0,
                "valid_scores":valid_scores
            }
        return {
            "final_score":None,"confidence":0.0,"flag":"judge_failed",
            "aggregation_method":None,"n_total_runs":1,"n_valid_runs":0,
            "n_used_in_final":0,"malformed_fraction":1.0,
            "flagged_outlier_scores":[],"median":None,"std":None,"valid_scores":[]
        }

    if malformed_fraction >= MALFORMED_RUN_FAILURE_FRACTION or len(valid_scores) < MIN_VALID_RUNS_FOR_SCORE:
        return {
            "final_score":None,"confidence":0.0,"flag":"judge_failed",
            "aggregation_method":None,"n_total_runs":n_total,"n_valid_runs":len(valid_scores),
            "n_used_in_final":0,"malformed_fraction":round(malformed_fraction,3),
            "flagged_outlier_scores":[],"median":None,"std":None,"valid_scores":valid_scores
        }

    median = statistics.median(valid_scores)
    std = statistics.pstdev(valid_scores) if len(valid_scores) > 1 else 0.0

    clean_scores, flagged_outliers = [], []
    for s in valid_scores:
        is_low_outlier = std > 0 and (median - s) > ZERO_OUTLIER_STD_THRESHOLD * std
        if s <= 1.0 and is_low_outlier:
            flagged_outliers.append(s)
        else:
            clean_scores.append(s)

    scores_for_agg = clean_scores if clean_scores else valid_scores
    median2 = statistics.median(scores_for_agg)
    std2 = statistics.pstdev(scores_for_agg) if len(scores_for_agg) > 1 else 0.0
    stability = max(0.0, 1 - std2/5.0)
    high_disagreement = std2 >= HIGH_DISAGREEMENT_STD

    if high_disagreement:
        final_score = median2
        method = "median_fallback_high_disagreement"
    else:
        weights = [1/(1 + GEMBA_LAMBDA*abs(s-median2)) for s in scores_for_agg]
        final_score = sum(w*s for w,s in zip(weights,scores_for_agg)) / sum(weights)
        method = "gemba_weighted_mean"

    flag = "ok"
    if flagged_outliers:
        flag = "zero_outliers_excluded"
    if high_disagreement:
        flag = "high_disagreement" if flag == "ok" else flag + "+high_disagreement"

    return {
        "final_score":round(final_score,3),
        "confidence":round(stability,3),
        "flag":flag,
        "aggregation_method":method,
        "n_total_runs":n_total,
        "n_valid_runs":len(valid_scores),
        "n_used_in_final":len(scores_for_agg),
        "malformed_fraction":round(malformed_fraction,3),
        "flagged_outlier_scores":flagged_outliers,
        "median":round(median2,3),
        "std":round(std2,3),
        "valid_scores":valid_scores,
    }

def _weighted_kappa(gold, pred, n_buckets=10):
    g = np.clip(np.round(gold),1,n_buckets).astype(int)
    p = np.clip(np.round(pred),1,n_buckets).astype(int)
    return float(cohen_kappa_score(g,p,weights="quadratic"))

def compute_agreement(gold, judged):
    gold_arr = np.array(gold,dtype=float)
    judged_arr = np.array(judged,dtype=float)
    if len(gold_arr) < 2:
        return {"pearson_r":np.nan,"spearman_rho":np.nan,"mae":np.nan,"weighted_kappa":np.nan,"n":len(gold_arr)}
    try: pearson = float(pearsonr(gold_arr,judged_arr)[0])
    except: pearson = np.nan
    try: spearman = float(spearmanr(gold_arr,judged_arr)[0])
    except: spearman = np.nan
    try: kappa = _weighted_kappa(gold_arr,judged_arr)
    except: kappa = np.nan
    return {
        "pearson_r":round(pearson,3),
        "spearman_rho":round(spearman,3),
        "mae":round(float(np.mean(np.abs(gold_arr-judged_arr))),3),
        "weighted_kappa":round(kappa,3),
        "n":len(gold_arr),
    }

In [9]:
# Cell 7 — Prompt builders P0–P4

COMMON_OUTPUT_SCHEMA = """
{
  "score": 1-10,
  "feedback": "brief explanation",
  "compliance": {
    "fully_translated": true,
    "target_language_mandarin": true,
    "simplified_chinese": true,
    "english_leakage": false,
    "pinyin_or_romanization": false,
    "cantonese_or_non_mandarin": false
  },
  "errors": [
    {
      "category": "mistranslation|omission|addition|entity|number|negation|terminology|fluency|grammar|language|script|source_leakage|format|other",
      "severity": "minor|major|critical",
      "description": "short description"
    }
  ]
}
""".strip()

def wrap_qwen(system_text, user_text):
    return (
        "<|im_start|>system\n" + system_text.strip() +
        "\n<|im_end|>\n<|im_start|>user\n" + user_text.strip() +
        "\n<|im_end|>\n<|im_start|>assistant\n/no_think\n"
    )

def build_p0_baseline(english, mandarin):
    system = '''
You are a strict professional translation evaluator.
Evaluate the Mandarin translation for accuracy, fluency, and fidelity.
Return ONLY valid JSON in this exact format:
{"score": <number from 1 to 10>, "feedback": "<short explanation>"}
Do not add any extra text.
Do not think aloud.
'''
    user = f"English source:\n{english}\n\nMandarin translation:\n{mandarin}"
    return wrap_qwen(system,user)

def build_p1_compliance(english, mandarin):
    system = f'''
You are a strict English-to-Mandarin translation evaluator.

FIRST check output compliance before judging fluency.

Required target:
- The source meaning must be translated, not copied.
- The output must be Mandarin Chinese.
- Use Simplified Chinese.
- Do not accept English/source leakage unless genuinely required.
- Do not accept Pinyin/romanization as a substitute for Chinese translation.
- Do not accept Cantonese-specific wording when standard Mandarin is required.

Mandatory score ceilings:
- Mostly untranslated / wrong target language: score <= 2
- Pinyin/romanization instead of Mandarin: score <= 2
- Predominantly Traditional Chinese when Simplified is required: score <= 4
- Major English/source leakage: score <= 4

After compliance, judge accuracy, fidelity, and fluency.

Return ONLY valid JSON using this schema:
{COMMON_OUTPUT_SCHEMA}

Do not add text outside JSON. Do not think aloud.
'''
    return wrap_qwen(system, f"English source:\n{english}\n\nCandidate Mandarin:\n{mandarin}")

def build_p2_semantic(english, mandarin):
    system = f'''
You are an English-to-Mandarin translation evaluator.

SEMANTIC FIDELITY HAS PRIORITY OVER FLUENCY.

Evaluate in this order:
1. Does the candidate preserve the complete meaning?
2. Check mistranslation, omission, unsupported addition, negation, entities, numbers,
   dates, quantities, modality, tense/aspect, conditions, and idiomatic meaning.
3. Check whether the candidate is actually Mandarin and fully translated.
4. Only after meaning is verified, evaluate naturalness, grammar, and fluency.

A fluent sentence with incorrect meaning must receive a low score.

Scoring guide:
9-10 = meaning fully preserved; natural Mandarin; at most negligible issues
7-8  = meaning preserved with minor non-critical issues
5-6  = noticeable semantic/language issues but core meaning partly preserved
3-4  = major mistranslation/omission/addition or serious language/script failure
1-2  = unusable, mostly untranslated, wrong meaning, or wrong language

Return ONLY valid JSON using this schema:
{COMMON_OUTPUT_SCHEMA}

Do not add text outside JSON. Do not think aloud.
'''
    return wrap_qwen(system, f"English source:\n{english}\n\nCandidate Mandarin:\n{mandarin}")

def build_p3_error_first(english, mandarin):
    system = f'''
You are a strict MQM-style English-to-Mandarin translation evaluator.

Use an ERROR-FIRST procedure:
A. Read the English source.
B. Read the Mandarin candidate.
C. Identify translation errors BEFORE selecting a score.
D. Classify each error.
E. Assign severity:
   - minor: limited problem; meaning remains clear and correct
   - major: substantial meaning/quality problem
   - critical: translation unusable or fundamentally wrong
F. Derive the final 1-10 score from the detected errors.

Error categories:
mistranslation, omission, addition, terminology, entity, number, negation,
fluency, grammar, language, script, source_leakage, format, other.

Do not start from an intuitive score and justify it afterward.

Return ONLY valid JSON using this schema:
{COMMON_OUTPUT_SCHEMA}

Do not add text outside JSON. Do not think aloud.
'''
    return wrap_qwen(system, f"English source:\n{english}\n\nCandidate Mandarin:\n{mandarin}")

def build_p4_combined(english, mandarin):
    system = f'''
You are a strict professional English-to-Mandarin translation evaluator.

STEP 1 — HARD COMPLIANCE
Check whether the candidate:
- is actually translated
- is Mandarin Chinese
- uses Simplified Chinese
- avoids inappropriate English/source leakage
- avoids Pinyin/romanization as a substitute for translation
- avoids Cantonese-specific language when standard Mandarin is required

Score ceilings:
- mostly untranslated / wrong language / Pinyin substitute: <= 2
- predominantly Traditional Chinese: <= 4
- major English/source leakage: <= 4

STEP 2 — SEMANTIC FIDELITY
Inspect mistranslation, omission, addition, negation, entities/names,
numbers/dates/quantities, terminology, modality/conditions, and idiomatic meaning.

STEP 3 — ERROR SEVERITY
Classify identified errors as minor, major, or critical.
A fluent but semantically wrong translation must score poorly.

STEP 4 — FLUENCY
Evaluate grammar and natural Mandarin only after compliance and meaning.

STEP 5 — FINAL SCORE
9-10 = fully correct meaning and compliant, natural Mandarin
7-8  = correct overall with minor errors
5-6  = partially correct with meaningful issues
3-4  = major errors or serious compliance failure
1-2  = unusable, fundamentally wrong, untranslated, or wrong language

Return ONLY valid JSON using this schema:
{COMMON_OUTPUT_SCHEMA}

Do not add text outside JSON. Do not think aloud.
'''
    return wrap_qwen(system, f"English source:\n{english}\n\nCandidate Mandarin:\n{mandarin}")

PROMPT_BUILDERS = {
    "P0_baseline":build_p0_baseline,
    "P1_compliance":build_p1_compliance,
    "P2_semantic":build_p2_semantic,
    "P3_error_first":build_p3_error_first,
    "P4_combined":build_p4_combined,
}
print(list(PROMPT_BUILDERS))

['P0_baseline', 'P1_compliance', 'P2_semantic', 'P3_error_first', 'P4_combined']


In [10]:
# Cell 8 — Challenge set

CHALLENGE_ROWS = [
{"sample_id":"CH01_correct","source_en":"Please close the window before you leave.","candidate_zh":"请在离开前关上窗户。","category":"correct","critical_failure":False,"expected_max_score":10},
{"sample_id":"CH02_valid_paraphrase","source_en":"The meeting has been postponed until Friday.","candidate_zh":"会议改到周五举行。","category":"valid_paraphrase","critical_failure":False,"expected_max_score":10},
{"sample_id":"CH03_minor_fluency","source_en":"I will call you when I arrive.","candidate_zh":"当我到达的时候，我会给你打电话。","category":"minor_fluency","critical_failure":False,"expected_max_score":8},
{"sample_id":"CH04_major_mistranslation","source_en":"The train leaves at six in the morning.","candidate_zh":"火车晚上六点出发。","category":"mistranslation","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH05_omission","source_en":"She bought apples, oranges, and two bottles of water.","candidate_zh":"她买了苹果和橙子。","category":"omission","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH06_addition","source_en":"He opened the door.","candidate_zh":"他生气地打开门，然后离开了房间。","category":"addition","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH07_wrong_entity","source_en":"Maria sent the report to David.","candidate_zh":"玛丽亚把报告发给了彼得。","category":"entity","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH08_wrong_number","source_en":"The package contains 15 tablets.","candidate_zh":"这个包装里有五片药片。","category":"number","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH09_negation","source_en":"Do not restart the computer.","candidate_zh":"请重新启动电脑。","category":"negation","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH10_untranslated_7_francs","source_en":"7 francs.","candidate_zh":"7 francs.","category":"untranslated","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH11_untranslated_sentence","source_en":"The doctor will see you tomorrow.","candidate_zh":"The doctor will see you tomorrow.","category":"untranslated","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH12_english_leakage","source_en":"Please upload the final report today.","candidate_zh":"请今天 upload 最终报告。","category":"source_leakage","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH13_traditional","source_en":"The weather is very good today.","candidate_zh":"今天天氣非常好。","category":"traditional_script","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH14_mixed_script","source_en":"Please confirm the information.","candidate_zh":"请確認这些信息。","category":"mixed_script","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH15_cantonese","source_en":"What are you doing?","candidate_zh":"你做緊乜嘢？","category":"cantonese","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH16_pinyin","source_en":"Thank you very much.","candidate_zh":"Feichang ganxie ni.","category":"pinyin","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH17_fluent_but_wrong","source_en":"The company reported a loss this quarter.","candidate_zh":"该公司本季度实现了盈利。","category":"fluent_but_wrong","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH18_wrong_date","source_en":"The deadline is October 12.","candidate_zh":"截止日期是10月21日。","category":"number","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH19_partial_translation","source_en":"Click Save, then restart the application.","candidate_zh":"点击“保存”，then restart the application.","category":"source_leakage","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH20_idiom","source_en":"Let's call it a day.","candidate_zh":"让我们称它为一天。","category":"idiom_mistranslation","critical_failure":True,"expected_max_score":4},
]
challenge_df = pd.DataFrame(CHALLENGE_ROWS)
challenge_df.to_csv(OUT/"challenge"/"challenge_set.csv", index=False)
display(challenge_df)

,sample_id,source_en,candidate_zh,category,critical_failure,expected_max_score
0,CH01_correct,Please close the window before you leave.,请在离开前关上窗户。,correct,False,10
1,CH02_valid_paraphrase,The meeting has been postponed until Friday.,会议改到周五举行。,valid_paraphrase,False,10
2,CH03_minor_fluency,I will call you when I arrive.,当我到达的时候，我会给你打电话。,minor_fluency,False,8
3,CH04_major_mistranslation,The train leaves at six in the morning.,火车晚上六点出发。,mistranslation,True,4
4,CH05_omission,"She bought apples, oranges, and two bottles of water.",她买了苹果和橙子。,omission,True,5
5,CH06_addition,He opened the door.,他生气地打开门，然后离开了房间。,addition,True,5
6,CH07_wrong_entity,Maria sent the report to David.,玛丽亚把报告发给了彼得。,entity,True,4
7,CH08_wrong_number,The package contains 15 tablets.,这个包装里有五片药片。,number,True,4
8,CH09_negation,Do not restart the computer.,请重新启动电脑。,negation,True,3
9,CH10_untranslated_7_francs,7 francs.,7 francs.,untranslated,True,2


In [12]:
# Cell 9 — Multi-prompt judge

class MultiPromptJudgeModel:
    def __init__(self, model_name, max_seq_length=4096, max_new_tokens=512, load_in_4bit=True):
        self.model_name = model_name
        self.max_seq_length = max_seq_length
        self.max_new_tokens = max_new_tokens
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=load_in_4bit,
        )
        FastLanguageModel.for_inference(self.model)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    def clean_output(self, text):
        text = re.sub(r"<think>.*?</think>","",text,flags=re.DOTALL)
        text = re.sub(r"</?think>","",text).strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?\s*","",text,flags=re.I)
            text = re.sub(r"\s*```$","",text)
        return text.strip()

    def _extract_json_object(self, text):
        try:
            obj = json.loads(text)
            return obj if isinstance(obj,dict) else None
        except:
            pass
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            try:
                obj = json.loads(text[start:end+1])
                return obj if isinstance(obj,dict) else None
            except:
                pass
        return None

    def parse_output(self, raw_output):
        cleaned = self.clean_output(raw_output)
        parsed = self._extract_json_object(cleaned)
        score, feedback, compliance, errors = None, cleaned, {}, []
        strict_json_ok = parsed is not None

        if parsed is not None:
            try: score = float(parsed.get("score"))
            except: score = None
            feedback = parsed.get("feedback", cleaned)
            compliance = parsed.get("compliance") or {}
            errors = parsed.get("errors") or []

        if score is None:
            match = re.search(r'(?<!\d)(10(?:\.0+)?|[1-9](?:\.\d+)?)(?!\d)', cleaned)
            if match:
                x = float(match.group(1))
                if SCORE_MIN <= x <= SCORE_MAX:
                    score = x

        score_in_range = score is not None and SCORE_MIN <= score <= SCORE_MAX
        if not isinstance(compliance,dict): compliance = {}
        if not isinstance(errors,list): errors = []

        return {
            "score":round(float(score),2) if score_in_range else None,
            "feedback":feedback,
            "strict_json_ok":strict_json_ok,
            "parse_ok":bool(score_in_range),
            "compliance":compliance,
            "errors":errors,
            "cleaned_output":cleaned,
        }

    def evaluate(self, english, mandarin, prompt_id, temperature=0.0, seed=None):
        if prompt_id not in PROMPT_BUILDERS:
            raise KeyError(prompt_id)
        if seed is not None:
            set_seed(seed)

        prompt = PROMPT_BUILDERS[prompt_id](english,mandarin)
        try:
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True,
                max_length=self.max_seq_length
            ).to(self.model.device)

            kwargs = dict(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=temperature > 0,
                renormalize_logits=True,
                use_cache=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
            if temperature > 0:
                kwargs["temperature"] = temperature

            with torch.no_grad():
                outputs = self.model.generate(**kwargs)

            generated_tokens = outputs[:, inputs.input_ids.shape[1]:]
            raw_output = self.tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
            parsed = self.parse_output(raw_output)

            return {
                **parsed,
                "success":parsed["parse_ok"],
                "error":None if parsed["parse_ok"] else "Could not parse valid 1-10 score",
                "raw_output":raw_output,
                "prompt_id":prompt_id,
                "temperature":temperature,
                "seed":seed,
            }
        except Exception as e:
            return {
                "score":None,"feedback":None,"strict_json_ok":False,"parse_ok":False,
                "compliance":{},"errors":[],"cleaned_output":None,"success":False,
                "error":str(e),"raw_output":None,"prompt_id":prompt_id,
                "temperature":temperature,"seed":seed,
            }

In [13]:
# Cell 10 — Load gold samples

gold_samples_all = load_gold_set(CONFIG["data"]["gold_set_csv_path"])
gold_validation = validate_gold_set(gold_samples_all)
gold_samples = [s for s in gold_samples_all if s.direction == CONFIG["data"]["direction"]]
gold_samples = gold_samples[:CONFIG["data"]["max_gold_samples"]]
if CONFIG["dry_run"]:
    gold_samples = gold_samples[:CONFIG["dry_run_n"]]

print("Validation:", gold_validation)
print("Selected EN→Mandarin gold samples:", len(gold_samples))
if len(gold_samples) == 0:
    raise RuntimeError("No EN_CMN gold samples found.")
save_json(gold_validation, OUT/"summaries"/"gold_set_validation.json")

Validation: {'n_total': 100, 'counts': {'EN_CMN': 50, 'EN_YUE': 50}, 'issues': []}
Selected EN→Mandarin gold samples: 50


In [14]:
# Cell 11 — Load Qwen3-8B once

judge = MultiPromptJudgeModel(
    model_name=CONFIG["judge"]["model_name"],
    max_seq_length=CONFIG["judge"]["max_seq_length"],
    max_new_tokens=CONFIG["judge"]["max_new_tokens"],
    load_in_4bit=CONFIG["judge"]["load_in_4bit"],
)
print("Loaded:", CONFIG["judge"]["model_name"])

==((====))==  Unsloth 2026.10.1: Fast Qwen3 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 6.878 GiB
no_split classes   : ['Qwen3DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 9.347 GiB requested
  cuda:0  budget  12.95 GiB  weights  3.562 GiB  free  9.391 GiB  reserve  9.347 GiB
  cuda:1  budget  12.98 GiB  weights  3.316 GiB  free  9.663 GiB  reserve  9.181 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.421 -> 12.979 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-8B


In [15]:
# Cell 12 — Unified experiment runner

def flatten_errors(errors):
    cats, sevs = [], []
    for e in errors or []:
        if isinstance(e,dict):
            c = str(e.get("category","")).strip().lower()
            s = str(e.get("severity","")).strip().lower()
            if c: cats.append(c)
            if s: sevs.append(s)
    return cats,sevs

def compliance_flags(c):
    c = c or {}
    return {
        "fully_translated":c.get("fully_translated"),
        "target_language_mandarin":c.get("target_language_mandarin"),
        "simplified_chinese":c.get("simplified_chinese"),
        "english_leakage":c.get("english_leakage"),
        "pinyin_or_romanization":c.get("pinyin_or_romanization"),
        "cantonese_or_non_mandarin":c.get("cantonese_or_non_mandarin"),
    }

def run_examples(examples, prompt_ids, n_runs, temperature, phase, dataset_name):
    rows, total, done = [], len(examples)*len(prompt_ids)*n_runs, 0

    for ex_idx, ex in enumerate(examples):
        for p_idx, prompt_id in enumerate(prompt_ids):
            for run_idx in range(n_runs):
                seed = (
                    CONFIG["judge"]["base_generation_seed"]
                    + ex_idx*1000 + p_idx*100 + run_idx
                )
                result = judge.evaluate(
                    english=ex["source_en"],
                    mandarin=ex["candidate_zh"],
                    prompt_id=prompt_id,
                    temperature=temperature,
                    seed=seed,
                )
                cats,sevs = flatten_errors(result.get("errors",[]))
                row = {
                    "dataset":dataset_name,"phase":phase,"sample_id":ex["sample_id"],
                    "prompt_id":prompt_id,"run_id":run_idx+1,
                    "temperature":temperature,"seed":seed,
                    "source_en":ex["source_en"],"candidate_zh":ex["candidate_zh"],
                    "gold_score":ex.get("gold_score"),"reference_zh":ex.get("reference_zh"),
                    "judge_score":result.get("score"),"parse_ok":result.get("parse_ok",False),
                    "strict_json_ok":result.get("strict_json_ok",False),
                    "feedback":result.get("feedback"),"raw_output":result.get("raw_output"),
                    "error_message":result.get("error"),
                    "error_categories":json.dumps(cats,ensure_ascii=False),
                    "error_severities":json.dumps(sevs,ensure_ascii=False),
                    "n_errors":len(cats),"has_major_error":"major" in sevs,
                    "has_critical_error":"critical" in sevs,
                    **compliance_flags(result.get("compliance",{})),
                }
                for key in ["category","critical_failure","expected_max_score"]:
                    if key in ex: row[key] = ex[key]
                rows.append(row)
                done += 1
                if done % 25 == 0 or done == total:
                    print(f"{phase} {dataset_name}: {done}/{total}")
    return pd.DataFrame(rows)

In [16]:
# Cell 13 — Unified records

gold_examples = [{
    "sample_id":s.sample_id,
    "source_en":s.source_en,
    "candidate_zh":s.mt_output,
    "gold_score":s.overall_gold,
    "reference_zh":s.reference_zh,
} for s in gold_samples]

challenge_examples = [dict(r) for r in CHALLENGE_ROWS]
print("Gold:",len(gold_examples),"Challenge:",len(challenge_examples))

Gold: 50 Challenge: 20


In [17]:
# Cell 14 — Phase A gold evaluation: P0–P4, temperature=0

phase_a_gold_raw = run_examples(
    gold_examples, PROMPT_IDS,
    CONFIG["judge"]["phase_a_runs"],
    CONFIG["judge"]["phase_a_temperature"],
    "A","gold"
)
phase_a_gold_raw.to_csv(OUT/"raw_runs"/"phase_a_gold_raw.csv",index=False)
display(phase_a_gold_raw.head())

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A gold: 25/250


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A gold: 50/250


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A gold: 225/250


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A gold: 250/250


,dataset,phase,sample_id,prompt_id,run_id,temperature,seed,source_en,candidate_zh,gold_score,reference_zh,judge_score,parse_ok,strict_json_ok,feedback,raw_output,error_message,error_categories,error_severities,n_errors,has_major_error,has_critical_error,fully_translated,target_language_mandarin,simplified_chinese,english_leakage,pinyin_or_romanization,cantonese_or_non_mandarin
0,gold,A,CMN_001,P0_baseline,1,0.0,5300,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,True,True,"Translation is accurate and fluent, with proper handling of titles and names. Minor improvement could be made in the order of names for consistency with English style.","</think>\n\n{""score"": 9, ""feedback"": ""Translation is accurate and fluent, with proper handling of titles and names. Minor improvement could be made in the order of names for co...",None,[],[],0,False,False,None,None,None,None,None,None
1,gold,A,CMN_001,P1_compliance,1,0.0,5400,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,10.0,True,False,"{""score"": 10, ""feedback"": ""Excellent translation."", ""compliance"": {""fully_translated"": true, ""target_language_mandarin"": true, ""simplified_chinese"": true, ""english_leakage"": fa...","{""score"": 10, ""feedback"": ""Excellent translation."", ""compliance"": {""fully_translated"": true, ""target_language_mandarin"": true, ""simplified_chinese"": true, ""english_leakage"": fa...",None,[],[],0,False,False,None,None,None,None,None,None
2,gold,A,CMN_001,P2_semantic,1,0.0,5500,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,True,True,The candidate translation preserves the complete meaning with only minor issues.,"</think>\n\n{\n ""score"": 9,\n ""feedback"": ""The candidate translation preserves the complete meaning with only minor issues."",\n ""compliance"": {\n ""fully_translated"": true...",None,"[""terminology""]","[""minor""]",1,False,False,True,True,True,False,False,False
3,gold,A,CMN_001,P3_error_first,1,0.0,5600,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,8.0,True,True,The translation is mostly accurate but contains some minor issues with terminology and fluency.,"</think>\n\n{\n ""score"": 8,\n ""feedback"": ""The translation is mostly accurate but contains some minor issues with terminology and fluency."",\n ""compliance"": {\n ""fully_tr...",None,"[""terminology"", ""fluency""]","[""minor"", ""minor""]",2,False,False,True,True,True,False,False,False
4,gold,A,CMN_001,P4_combined,1,0.0,5700,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,True,True,Accurate and fluent translation with minor improvements possible for terminology consistency,"</think>\n\n{\n ""score"": 9,\n ""feedback"": ""Accurate and fluent translation with minor improvements possible for terminology consistency"",\n ""compliance"": {\n ""fully_trans...",None,"[""terminology""]","[""minor""]",1,False,False,True,True,True,False,False,False


In [18]:
# Cell 15 — Aggregate runs

def aggregate_run_dataframe(raw_df):
    out = []
    group_cols = ["dataset","phase","sample_id","prompt_id"]

    for keys,g in raw_df.groupby(group_cols,dropna=False):
        dataset,phase,sample_id,prompt_id = keys
        rr = [
            RunResult(
                score=safe_float(row["judge_score"]),
                parse_ok=bool(row["parse_ok"]),
                raw_response=row.get("raw_output")
            )
            for _,row in g.iterrows()
        ]
        agg = aggregate_runs(rr)
        first = g.iloc[0]
        valid = [float(x) for x in g.loc[g["parse_ok"]==True,"judge_score"].dropna().tolist()]
        exact = len(valid)>0 and len(set(round(x,6) for x in valid))==1
        score_range = max(valid)-min(valid) if valid else np.nan
        raw_std = statistics.pstdev(valid) if len(valid)>1 else (0.0 if len(valid)==1 else np.nan)

        row = {
            "dataset":dataset,"phase":phase,"sample_id":sample_id,"prompt_id":prompt_id,
            "source_en":first["source_en"],"candidate_zh":first["candidate_zh"],
            "gold_score":first.get("gold_score"),"reference_zh":first.get("reference_zh"),
            "final_score":agg["final_score"],"judge_confidence":agg["confidence"],
            "judge_flag":agg["flag"],"aggregation_method":agg.get("aggregation_method"),
            "n_total_runs":agg["n_total_runs"],"n_valid_runs":agg["n_valid_runs"],
            "malformed_fraction":agg.get("malformed_fraction"),
            "valid_scores":json.dumps(agg.get("valid_scores",[])),
            "run_mean":np.mean(valid) if valid else np.nan,
            "run_std":raw_std,"run_range":score_range,"exact_agreement":exact,
            "strict_json_success_rate":float(g["strict_json_ok"].mean()),
            "parse_success_rate":float(g["parse_ok"].mean()),
        }
        for col in ["category","critical_failure","expected_max_score"]:
            if col in g.columns: row[col] = first.get(col)
        out.append(row)
    return pd.DataFrame(out)

phase_a_gold_agg = aggregate_run_dataframe(phase_a_gold_raw)
phase_a_gold_agg.to_csv(OUT/"aggregated"/"phase_a_gold_aggregated.csv",index=False)
display(phase_a_gold_agg.head())

,dataset,phase,sample_id,prompt_id,source_en,candidate_zh,gold_score,reference_zh,final_score,judge_confidence,judge_flag,aggregation_method,n_total_runs,n_valid_runs,malformed_fraction,valid_scores,run_mean,run_std,run_range,exact_agreement,strict_json_success_rate,parse_success_rate
0,gold,A,CMN_001,P0_baseline,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,1.0,ok,single_run,1,1,0.0,[9.0],9.0,0.0,0.0,True,1.0,1.0
1,gold,A,CMN_001,P1_compliance,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,10.0,1.0,ok,single_run,1,1,0.0,[10.0],10.0,0.0,0.0,True,0.0,1.0
2,gold,A,CMN_001,P2_semantic,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,1.0,ok,single_run,1,1,0.0,[9.0],9.0,0.0,0.0,True,1.0,1.0
3,gold,A,CMN_001,P3_error_first,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,8.0,1.0,ok,single_run,1,1,0.0,[8.0],8.0,0.0,0.0,True,1.0,1.0
4,gold,A,CMN_001,P4_combined,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,1.0,ok,single_run,1,1,0.0,[9.0],9.0,0.0,0.0,True,1.0,1.0


In [19]:
# Cell 16 — Gold validity metrics

def gold_prompt_metrics(agg_df):
    rows = []
    for prompt_id,g in agg_df.groupby("prompt_id"):
        usable = g[g["final_score"].notna() & g["gold_score"].notna() & (g["judge_flag"]!="judge_failed")]
        agreement = compute_agreement(
            usable["gold_score"].astype(float).tolist(),
            usable["final_score"].astype(float).tolist()
        )
        rows.append({
            "prompt_id":prompt_id,
            "n_total":len(g),
            "n_usable":len(usable),
            "coverage":round(len(usable)/len(g),3) if len(g) else 0.0,
            "strict_json_success":round(float(g["strict_json_success_rate"].mean()),3),
            "parse_success":round(float(g["parse_success_rate"].mean()),3),
            **agreement,
        })
    return pd.DataFrame(rows).sort_values(["spearman_rho","mae"],ascending=[False,True])

phase_a_gold_metrics = gold_prompt_metrics(phase_a_gold_agg)
phase_a_gold_metrics.to_csv(OUT/"tables"/"phase_a_gold_validity_metrics.csv",index=False)
display(phase_a_gold_metrics)

,prompt_id,n_total,n_usable,coverage,strict_json_success,parse_success,pearson_r,spearman_rho,mae,weighted_kappa,n
0,P0_baseline,50,50,1.0,0.72,1.0,0.731,0.701,0.694,0.678,50
1,P1_compliance,50,50,1.0,0.84,1.0,0.626,0.690,0.714,0.611,50
4,P4_combined,50,50,1.0,1.00,1.0,0.623,0.657,1.034,0.480,50
2,P2_semantic,50,50,1.0,1.00,1.0,0.592,0.603,0.758,0.526,50
3,P3_error_first,50,50,1.0,1.00,1.0,0.528,0.591,1.922,0.245,50


In [20]:
# Cell 17 — Phase A challenge set

phase_a_challenge_raw = run_examples(
    challenge_examples, PROMPT_IDS,
    CONFIG["judge"]["phase_a_runs"],
    CONFIG["judge"]["phase_a_temperature"],
    "A","challenge"
)
phase_a_challenge_raw.to_csv(OUT/"raw_runs"/"phase_a_challenge_raw.csv",index=False)
phase_a_challenge_agg = aggregate_run_dataframe(phase_a_challenge_raw)
phase_a_challenge_agg.to_csv(OUT/"aggregated"/"phase_a_challenge_aggregated.csv",index=False)
display(phase_a_challenge_agg[["sample_id","category","prompt_id","final_score","expected_max_score"]].head(30))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A challenge: 25/100


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A challenge: 50/100


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A challenge: 75/100


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

A challenge: 100/100


,sample_id,category,prompt_id,final_score,expected_max_score
0,CH01_correct,correct,P0_baseline,10.0,10
1,CH01_correct,correct,P1_compliance,10.0,10
2,CH01_correct,correct,P2_semantic,10.0,10
3,CH01_correct,correct,P3_error_first,10.0,10
4,CH01_correct,correct,P4_combined,10.0,10
5,CH02_valid_paraphrase,valid_paraphrase,P0_baseline,10.0,10
6,CH02_valid_paraphrase,valid_paraphrase,P1_compliance,10.0,10
7,CH02_valid_paraphrase,valid_paraphrase,P2_semantic,10.0,10
8,CH02_valid_paraphrase,valid_paraphrase,P3_error_first,8.0,10
9,CH02_valid_paraphrase,valid_paraphrase,P4_combined,9.0,10


In [21]:
# Cell 18 — Challenge metrics

FAILURE_CATEGORY_ALIASES = {
    "mistranslation":{"mistranslation","semantic","accuracy"},
    "idiom_mistranslation":{"mistranslation","semantic","accuracy"},
    "fluent_but_wrong":{"mistranslation","semantic","accuracy"},
    "omission":{"omission"},
    "addition":{"addition"},
    "entity":{"entity","mistranslation"},
    "number":{"number","mistranslation"},
    "negation":{"negation","mistranslation"},
    "untranslated":{"language","source_leakage","format"},
    "source_leakage":{"source_leakage","language"},
    "traditional_script":{"script","language"},
    "mixed_script":{"script","language"},
    "cantonese":{"language","terminology"},
    "pinyin":{"language","source_leakage","format"},
}

def parse_list_cell(x):
    try:
        v = json.loads(x)
        return [str(a).lower() for a in v] if isinstance(v,list) else []
    except:
        return []

def challenge_metrics(raw_df,agg_df):
    rows = []
    for prompt_id in sorted(agg_df["prompt_id"].unique()):
        a = agg_df[agg_df["prompt_id"]==prompt_id].copy()
        r = raw_df[raw_df["prompt_id"]==prompt_id].copy()
        critical = a[a["critical_failure"]==True]
        noncritical = a[a["critical_failure"]==False]

        critical_recall = float(
            (critical["final_score"].notna() &
             (critical["final_score"] <= critical["expected_max_score"])).mean()
        ) if len(critical) else np.nan

        false_positive = float(
            (noncritical["final_score"] <= 6).mean()
        ) if len(noncritical) else np.nan

        hits = []
        for _,row in r[r["critical_failure"]==True].iterrows():
            cat = str(row["category"]).lower()
            cats = set(parse_list_cell(row["error_categories"]))
            expected = FAILURE_CATEGORY_ALIASES.get(cat,set())

            flag_hit = False
            if cat == "untranslated":
                flag_hit = row.get("fully_translated") is False or row.get("target_language_mandarin") is False
            elif cat == "source_leakage":
                flag_hit = row.get("english_leakage") is True
            elif cat in {"traditional_script","mixed_script"}:
                flag_hit = row.get("simplified_chinese") is False
            elif cat == "pinyin":
                flag_hit = row.get("pinyin_or_romanization") is True
            elif cat == "cantonese":
                flag_hit = row.get("cantonese_or_non_mandarin") is True

            hits.append(bool(cats.intersection(expected)) or bool(flag_hit))

        rows.append({
            "prompt_id":prompt_id,
            "n_challenge":len(a),
            "n_critical":len(critical),
            "critical_failure_score_recall":round(critical_recall,3),
            "structured_failure_recall":round(float(np.mean(hits)),3) if hits else np.nan,
            "false_positive_rate":round(false_positive,3),
            "mean_critical_score":round(float(critical["final_score"].mean()),3),
            "mean_noncritical_score":round(float(noncritical["final_score"].mean()),3),
            "strict_json_success":round(float(r["strict_json_ok"].mean()),3),
            "parse_success":round(float(r["parse_ok"].mean()),3),
        })
    return pd.DataFrame(rows).sort_values(
        ["critical_failure_score_recall","structured_failure_recall"],ascending=False
    )

phase_a_challenge_metrics = challenge_metrics(phase_a_challenge_raw,phase_a_challenge_agg)
phase_a_challenge_metrics.to_csv(OUT/"tables"/"phase_a_challenge_metrics.csv",index=False)
display(phase_a_challenge_metrics)

,prompt_id,n_challenge,n_critical,critical_failure_score_recall,structured_failure_recall,false_positive_rate,mean_critical_score,mean_noncritical_score,strict_json_success,parse_success
1,P1_compliance,20,17,0.529,0.000,0.0,4.353,10.000,0.00,1.0
4,P4_combined,20,17,0.412,0.176,0.0,4.471,9.667,0.20,1.0
3,P3_error_first,20,17,0.353,0.235,0.0,4.647,9.333,0.40,1.0
2,P2_semantic,20,17,0.294,0.235,0.0,5.471,10.000,0.35,1.0
0,P0_baseline,20,17,0.235,0.000,0.0,6.000,10.000,0.00,1.0


In [22]:
# Cell 19 — Rank and select Phase B prompts

def minmax(series,higher_is_better=True):
    s = pd.to_numeric(series,errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series([0.0]*len(s),index=s.index)
    lo,hi = s.min(),s.max()
    out = pd.Series([1.0]*len(s),index=s.index) if math.isclose(float(lo),float(hi)) else (s-lo)/(hi-lo)
    return out if higher_is_better else 1-out

phase_a_rank = phase_a_gold_metrics.merge(
    phase_a_challenge_metrics,on="prompt_id",suffixes=("_gold","_challenge")
)
phase_a_rank["selection_score"] = (
    0.25*minmax(phase_a_rank["spearman_rho"],True)
    +0.20*minmax(phase_a_rank["weighted_kappa"],True)
    +0.15*minmax(phase_a_rank["mae"],False)
    +0.20*minmax(phase_a_rank["critical_failure_score_recall"],True)
    +0.10*minmax(phase_a_rank["structured_failure_recall"],True)
    +0.10*minmax(phase_a_rank["strict_json_success_gold"],True)
)
phase_a_rank = phase_a_rank.sort_values("selection_score",ascending=False).reset_index(drop=True)

selected_phase_b = phase_a_rank.head(
    CONFIG["selection"]["top_prompts_for_phase_b"]
)["prompt_id"].tolist()

if "P0_baseline" not in selected_phase_b:
    selected_phase_b = ["P0_baseline"] + selected_phase_b
selected_phase_b = list(dict.fromkeys(selected_phase_b))

phase_a_rank.to_csv(OUT/"tables"/"phase_a_prompt_ranking.csv",index=False)
save_json({"selected_phase_b_prompts":selected_phase_b},OUT/"summaries"/"phase_b_selection.json")
display(phase_a_rank)
print("Phase B:",selected_phase_b)

,prompt_id,n_total,n_usable,coverage,strict_json_success_gold,parse_success_gold,pearson_r,spearman_rho,mae,weighted_kappa,n,n_challenge,n_critical,critical_failure_score_recall,structured_failure_recall,false_positive_rate,mean_critical_score,mean_noncritical_score,strict_json_success_challenge,parse_success_challenge,selection_score
0,P1_compliance,50,50,1.0,0.84,1.0,0.626,0.690,0.714,0.611,50,20,17,0.529,0.000,0.0,4.353,10.000,0.00,1.0,0.784467
1,P4_combined,50,50,1.0,1.00,1.0,0.623,0.657,1.034,0.480,50,20,17,0.412,0.176,0.0,4.471,9.667,0.20,1.0,0.662316
2,P0_baseline,50,50,1.0,0.72,1.0,0.731,0.701,0.694,0.678,50,20,17,0.235,0.000,0.0,6.000,10.000,0.00,1.0,0.600000
3,P2_semantic,50,50,1.0,1.00,1.0,0.592,0.603,0.758,0.526,50,20,17,0.294,0.235,0.0,5.471,10.000,0.35,1.0,0.539383
4,P3_error_first,50,50,1.0,1.00,1.0,0.528,0.591,1.922,0.245,50,20,17,0.353,0.235,0.0,4.647,9.333,0.40,1.0,0.280272


Phase B: ['P1_compliance', 'P4_combined', 'P0_baseline']


In [23]:
# Cell 20 — Phase B repeated runs

phase_b_gold_raw = run_examples(
    gold_examples, selected_phase_b,
    CONFIG["judge"]["phase_b_runs"],
    CONFIG["judge"]["phase_b_temperature"],
    "B","gold"
)
phase_b_gold_raw.to_csv(OUT/"raw_runs"/"phase_b_gold_raw.csv",index=False)
phase_b_gold_agg = aggregate_run_dataframe(phase_b_gold_raw)
phase_b_gold_agg.to_csv(OUT/"aggregated"/"phase_b_gold_aggregated.csv",index=False)
display(phase_b_gold_agg.head())

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

B gold: 25/750


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

B gold: 50/750


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

B gold: 725/750


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

B gold: 750/750


,dataset,phase,sample_id,prompt_id,source_en,candidate_zh,gold_score,reference_zh,final_score,judge_confidence,judge_flag,aggregation_method,n_total_runs,n_valid_runs,malformed_fraction,valid_scores,run_mean,run_std,run_range,exact_agreement,strict_json_success_rate,parse_success_rate
0,gold,B,CMN_001,P0_baseline,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,1.0,ok,gemba_weighted_mean,5,5,0.0,"[9.0, 9.0, 9.0, 9.0, 9.0]",9.0,0.0,0.0,True,1.0,1.0
1,gold,B,CMN_001,P1_compliance,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,10.0,1.0,ok,gemba_weighted_mean,5,5,0.0,"[10.0, 10.0, 10.0, 10.0, 10.0]",10.0,0.0,0.0,True,0.0,1.0
2,gold,B,CMN_001,P4_combined,Turkish diva Sezen Aksu performed with the Italian tenor Alessandro Safina and Greek singer Haris Alexiou.,土耳其女歌手塞泽尔·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修共同表演。,9.8,土耳其歌后塞泽·阿克苏与意大利男高音亚历山德罗·萨菲纳和希腊歌手哈里斯·阿莱克修同台演出。,9.0,1.0,ok,gemba_weighted_mean,5,5,0.0,"[9.0, 9.0, 9.0, 9.0, 9.0]",9.0,0.0,0.0,True,1.0,1.0
3,gold,B,CMN_002,P0_baseline,Another disadvantage of zoom lenses is that the maximum aperture (the speed) of the lens is usually lower.,变焦镜头的另一个缺点是，镜头的最大光圈（即速度）通常较低。,9.3,变焦镜头的另一个缺点是其最大光圈（镜头速度）通常较小。,9.0,1.0,ok,gemba_weighted_mean,5,5,0.0,"[9.0, 9.0, 9.0, 9.0, 9.0]",9.0,0.0,0.0,True,0.4,1.0
4,gold,B,CMN_002,P1_compliance,Another disadvantage of zoom lenses is that the maximum aperture (the speed) of the lens is usually lower.,变焦镜头的另一个缺点是，镜头的最大光圈（即速度）通常较低。,9.3,变焦镜头的另一个缺点是其最大光圈（镜头速度）通常较小。,9.0,1.0,ok,gemba_weighted_mean,5,5,0.0,"[9.0, 9.0, 9.0, 9.0, 9.0]",9.0,0.0,0.0,True,1.0,1.0


In [24]:
# Cell 21 — Consistency metrics

def consistency_metrics(agg_df):
    rows = []
    for prompt_id,g in agg_df.groupby("prompt_id"):
        usable = g[g["n_valid_runs"]>0].copy()
        high = (
            (usable["run_range"] >= CONFIG["consistency"]["high_disagreement_range"])
            | (usable["run_std"] >= CONFIG["consistency"]["high_disagreement_std"])
        )
        rows.append({
            "prompt_id":prompt_id,
            "n_samples":len(g),
            "mean_run_std":round(float(usable["run_std"].mean()),3),
            "median_run_std":round(float(usable["run_std"].median()),3),
            "mean_run_range":round(float(usable["run_range"].mean()),3),
            "median_run_range":round(float(usable["run_range"].median()),3),
            "exact_agreement_rate":round(float(usable["exact_agreement"].mean()),3),
            "high_disagreement_rate":round(float(high.mean()),3),
            "mean_judge_confidence":round(float(usable["judge_confidence"].mean()),3),
            "mean_parse_success":round(float(usable["parse_success_rate"].mean()),3),
            "mean_strict_json_success":round(float(usable["strict_json_success_rate"].mean()),3),
        })
    return pd.DataFrame(rows).sort_values(["mean_run_std","mean_run_range"])

phase_b_consistency = consistency_metrics(phase_b_gold_agg)
phase_b_consistency.to_csv(OUT/"tables"/"phase_b_consistency_metrics.csv",index=False)
display(phase_b_consistency)

,prompt_id,n_samples,mean_run_std,median_run_std,mean_run_range,median_run_range,exact_agreement_rate,high_disagreement_rate,mean_judge_confidence,mean_parse_success,mean_strict_json_success
2,P4_combined,50,0.000,0.0,0.00,0.0,1.00,0.0,1.000,1.0,1.000
0,P0_baseline,50,0.055,0.0,0.12,0.0,0.88,0.0,0.989,1.0,0.756
1,P1_compliance,50,0.055,0.0,0.12,0.0,0.88,0.0,0.989,1.0,0.828


In [25]:
# Cell 22 — Phase B validity after aggregation

phase_b_gold_metrics = gold_prompt_metrics(phase_b_gold_agg)
phase_b_gold_metrics.to_csv(OUT/"tables"/"phase_b_gold_validity_metrics.csv",index=False)
display(phase_b_gold_metrics)

,prompt_id,n_total,n_usable,coverage,strict_json_success,parse_success,pearson_r,spearman_rho,mae,weighted_kappa,n
0,P0_baseline,50,50,1.0,0.756,1.0,0.721,0.719,0.704,0.656,50
1,P1_compliance,50,50,1.0,0.828,1.0,0.636,0.707,0.668,0.613,50
2,P4_combined,50,50,1.0,1.000,1.0,0.623,0.657,1.034,0.480,50


In [26]:
# Cell 23 — Final comparison table

final_table = (
    phase_a_gold_metrics[
        ["prompt_id","spearman_rho","pearson_r","mae","weighted_kappa",
         "coverage","strict_json_success","parse_success"]
    ].rename(columns={
        "spearman_rho":"A_spearman","pearson_r":"A_pearson","mae":"A_mae",
        "weighted_kappa":"A_kappa","coverage":"A_coverage",
        "strict_json_success":"A_json_success","parse_success":"A_parse_success"
    })
    .merge(
        phase_a_challenge_metrics[
            ["prompt_id","critical_failure_score_recall","structured_failure_recall","false_positive_rate"]
        ],on="prompt_id",how="left"
    )
    .merge(phase_b_consistency,on="prompt_id",how="left")
    .merge(
        phase_b_gold_metrics[["prompt_id","spearman_rho","mae","weighted_kappa"]]
        .rename(columns={"spearman_rho":"B_spearman","mae":"B_mae","weighted_kappa":"B_kappa"}),
        on="prompt_id",how="left"
    )
)

final_table["final_rank_score"] = (
    0.20*minmax(final_table["A_spearman"],True)
    +0.15*minmax(final_table["A_kappa"],True)
    +0.10*minmax(final_table["A_mae"],False)
    +0.20*minmax(final_table["critical_failure_score_recall"],True)
    +0.10*minmax(final_table["structured_failure_recall"],True)
    +0.10*minmax(final_table["mean_run_std"],False)
    +0.05*minmax(final_table["mean_run_range"],False)
    +0.05*minmax(final_table["high_disagreement_rate"],False)
    +0.05*minmax(final_table["A_json_success"],True)
)

final_table = final_table.sort_values("final_rank_score",ascending=False).reset_index(drop=True)
final_table.to_csv(OUT/"tables"/"final_prompt_comparison.csv",index=False)
display(final_table)

,prompt_id,A_spearman,A_pearson,A_mae,A_kappa,A_coverage,A_json_success,A_parse_success,critical_failure_score_recall,structured_failure_recall,false_positive_rate,n_samples,mean_run_std,median_run_std,mean_run_range,median_run_range,exact_agreement_rate,high_disagreement_rate,mean_judge_confidence,mean_parse_success,mean_strict_json_success,B_spearman,B_mae,B_kappa,final_rank_score
0,P4_combined,0.657,0.623,1.034,0.480,1.0,1.00,1.0,0.412,0.176,0.0,50.0,0.000,0.0,0.00,0.0,1.00,0.0,1.000,1.0,1.000,0.657,1.034,0.480,0.669023
1,P1_compliance,0.690,0.626,0.714,0.611,1.0,0.84,1.0,0.529,0.000,0.0,50.0,0.055,0.0,0.12,0.0,0.88,0.0,0.989,1.0,0.828,0.707,0.668,0.613,0.626590
2,P0_baseline,0.701,0.731,0.694,0.678,1.0,0.72,1.0,0.235,0.000,0.0,50.0,0.055,0.0,0.12,0.0,0.88,0.0,0.989,1.0,0.756,0.719,0.704,0.656,0.450000
3,P2_semantic,0.603,0.592,0.758,0.526,1.0,1.00,1.0,0.294,0.235,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P3_error_first,0.591,0.528,1.922,0.245,1.0,1.00,1.0,0.353,0.235,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# Cell 24 — Per-category challenge table

challenge_category_table = (
    phase_a_challenge_agg
    .groupby(["prompt_id","category"],as_index=False)
    .agg(
        n=("sample_id","count"),
        mean_score=("final_score","mean"),
        max_score=("final_score","max"),
        expected_max_score=("expected_max_score","mean"),
    )
)
challenge_category_table.to_csv(OUT/"tables"/"challenge_by_category.csv",index=False)
display(challenge_category_table)

,prompt_id,category,n,mean_score,max_score,expected_max_score
0,P0_baseline,addition,1,5.0,5.0,5.0
1,P0_baseline,cantonese,1,5.0,5.0,3.0
2,P0_baseline,correct,1,10.0,10.0,10.0
3,P0_baseline,entity,1,5.0,5.0,4.0
4,P0_baseline,fluent_but_wrong,1,1.0,1.0,3.0
...,...,...,...,...,...,...
80,P4_combined,pinyin,1,5.0,5.0,2.0
81,P4_combined,source_leakage,2,4.5,7.0,4.0
82,P4_combined,traditional_script,1,10.0,10.0,4.0
83,P4_combined,untranslated,2,2.0,2.0,2.0


In [28]:
# Cell 25 — Consolidated JSON summary

summary = {
    "experiment_name":CONFIG["experiment_name"],
    "judge_model":CONFIG["judge"]["model_name"],
    "direction":CONFIG["data"]["direction"],
    "n_gold":len(gold_examples),
    "n_challenge":len(challenge_examples),
    "prompt_ids":PROMPT_IDS,
    "phase_a":{
        "temperature":CONFIG["judge"]["phase_a_temperature"],
        "runs":CONFIG["judge"]["phase_a_runs"],
        "gold_metrics":phase_a_gold_metrics.to_dict(orient="records"),
        "challenge_metrics":phase_a_challenge_metrics.to_dict(orient="records"),
        "ranking":phase_a_rank.to_dict(orient="records"),
    },
    "phase_b":{
        "temperature":CONFIG["judge"]["phase_b_temperature"],
        "runs":CONFIG["judge"]["phase_b_runs"],
        "selected_prompts":selected_phase_b,
        "consistency":phase_b_consistency.to_dict(orient="records"),
        "gold_metrics":phase_b_gold_metrics.to_dict(orient="records"),
    },
    "final_comparison":final_table.to_dict(orient="records"),
}
save_json(summary,OUT/"summaries"/"experiment_summary.json")
print(json.dumps(summary,ensure_ascii=False,indent=2,default=str)[:8000])

{
  "experiment_name": "sprint53_task2_multiprompt_judge",
  "judge_model": "Qwen/Qwen3-8B",
  "direction": "EN_CMN",
  "n_gold": 50,
  "n_challenge": 20,
  "prompt_ids": [
    "P0_baseline",
    "P1_compliance",
    "P2_semantic",
    "P3_error_first",
    "P4_combined"
  ],
  "phase_a": {
    "temperature": 0.0,
    "runs": 1,
    "gold_metrics": [
      {
        "prompt_id": "P0_baseline",
        "n_total": 50,
        "n_usable": 50,
        "coverage": 1.0,
        "strict_json_success": 0.72,
        "parse_success": 1.0,
        "pearson_r": 0.731,
        "spearman_rho": 0.701,
        "mae": 0.694,
        "weighted_kappa": 0.678,
        "n": 50
      },
      {
        "prompt_id": "P1_compliance",
        "n_total": 50,
        "n_usable": 50,
        "coverage": 1.0,
        "strict_json_success": 0.84,
        "parse_success": 1.0,
        "pearson_r": 0.626,
        "spearman_rho": 0.69,
        "mae": 0.714,
        "weighted_kappa": 0.611,
        "n": 50
      },
  

In [29]:
# Cell 26 — Artifact manifest

manifest = []
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        manifest.append({"file":str(path.relative_to(OUT)),"size_bytes":path.stat().st_size})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT/"artifact_manifest.csv",index=False)
display(manifest_df)

print("Recommended report inputs:")
print("tables/final_prompt_comparison.csv")
print("tables/phase_a_gold_validity_metrics.csv")
print("tables/phase_a_challenge_metrics.csv")
print("tables/phase_b_consistency_metrics.csv")
print("tables/challenge_by_category.csv")
print("summaries/experiment_summary.json")
print("raw_runs/phase_b_gold_raw.csv")

# Interpretation: 
# Use four dimensions when choosing the final prompt:
# 1. **Validity against humans:** Spearman, Pearson, weighted κ, MAE.
# 2. **Failure detection:** challenge-set critical-failure recall and structured-error recall.
# 3. **Repeated-run consistency:** run SD, score range, exact agreement, high-disagreement rate.
# 4. **Output reliability:** strict JSON success and parse success.
# Do not choose the winning prompt from one metric alone. A useful judge should be both valid and stable, while detecting obvious semantic, language, and script failures.

,file,size_bytes
0,aggregated/phase_a_challenge_aggregated.csv,19344
1,aggregated/phase_a_gold_aggregated.csv,111087
2,aggregated/phase_b_gold_aggregated.csv,71630
3,challenge/challenge_set.csv,2107
4,raw_runs/phase_a_challenge_raw.csv,164135
5,raw_runs/phase_a_gold_raw.csv,316881
6,raw_runs/phase_b_gold_raw.csv,886894
7,summaries/experiment_summary.json,13842
8,summaries/gold_set_validation.json,90
9,summaries/phase_b_selection.json,97


Recommended report inputs:
tables/final_prompt_comparison.csv
tables/phase_a_gold_validity_metrics.csv
tables/phase_a_challenge_metrics.csv
tables/phase_b_consistency_metrics.csv
tables/challenge_by_category.csv
summaries/experiment_summary.json
raw_runs/phase_b_gold_raw.csv
